## 1. Importing Libraries

In [23]:
import pandas as pd
import numpy as np
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
import torch.nn.functional as F
from transformers import EarlyStoppingCallback

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Selected device: {device}")

Selected device: cuda


## 2. Loading and Preprocessing Data

In [ ]:
def clean_text(text):
    text = text.replace("@anonymized_account", "")
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'(@\w+)', '', text)
    text = re.sub(r'(\w)\1{2,}', r'\1\1', text) 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df = pd.read_csv('hate_train.csv')
train_df['sentence'] = train_df['sentence'].apply(clean_text)

with open('hate_test_data.txt', 'r', encoding='utf-8') as f:
    test_texts = [clean_text(line.strip()) for line in f.readlines()]

test_df = pd.DataFrame({'sentence': test_texts})

## 3. Tokenization and Dataset Split

In [25]:
model_name = "sdadas/polish-roberta-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding="max_length", truncation=True, max_length=512)

tokenized_train = train_ds.map(tokenize_function, batched=True)
tokenized_test = test_ds.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column('label', 'labels')
tokenized_train = tokenized_train.class_encode_column("labels")

split = tokenized_train.train_test_split(test_size=0.1, stratify_by_column="labels", seed=42)
train_dataset = split['train']
eval_dataset = split['test']

Map:   0%|          | 0/10041 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/10041 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/10041 [00:00<?, ? examples/s]

## 4. Class Imbalance Mitigation

In [26]:
y_train = train_dataset['labels']
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.5):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        weights = class_weights_tensor.to(model.device)
        loss_fct = FocalLoss(alpha=weights, gamma=2.5)
        
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1_macro = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1_macro': f1_macro}

## 5. Model Initialization and Training

In [27]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    weight_decay=0.08,
    warmup_ratio=0.15,
    lr_scheduler_type="cosine_with_restarts",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    fp16=True, 
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()
val_results = trainer.evaluate()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: sdadas/polish-roberta-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.877534,0.161301,0.084577,0.077982
2,0.613392,0.167346,0.742289,0.602487
3,0.481728,0.160321,0.714428,0.580398


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.481728,0.167346,3,0.742289,0.602487


## 6. Test Set Prediction

In [28]:
predictions = trainer.predict(tokenized_test)
preds_raw = predictions.predictions.argmax(-1)

hate_keywords = {
    'kurwa', 'kurwy', 'skurwysyn', 'skurwiel', 'jebać', 'jeban', 'zjeb', 'zjebać', 'gnida',
    'szmata', 'szmaty', 'pedał', 'pedały', 'ciota', 'pizda', 'chuj', 'chuju', 'chujek',
    'debil', 'debile', 'idiota', 'kretyn', 'pajac', 'miernota', 'gnojek', 'łachudra',
    'wypierdalaj', 'wypierdolić', 'spierdalaj', 'spierdolić', 'pierdol', 'rozpierdalać',
    'żyd', 'żydzi', 'żydostwo', 'lewacka', 'lewus', 'pisowska', 'pisior', 'kaczystan',
    'zoofil', 'gwałciciel', 'dziwka', 'szmata', 'cwel', 'frajer', 'śmieć', 'gówno',
    'napluć', 'w gębę', 'w mordę', 'do gazu', 'powiesił', 'samobój'
}

def post_process(predictions, texts, raw_logits=None):
    final = []
    for i, (pred, text) in enumerate(zip(predictions, texts)):
        lower = text.lower()
        
        if any(kw in lower for kw in hate_keywords):
            final.append(1)
            continue
            
        weak_signals = ['debil', 'idiota', 'pajac', 'kretyn', 'gnida', 'szmata', 'łżesz', 'kłamliwa']
        if any(sig in lower for sig in weak_signals) and pred == 0:
            final.append(1)
        else:
            final.append(pred)
    return final

preds_processed = post_process(preds_raw, test_texts)

pd.DataFrame(preds_processed).to_csv('pred.csv', index=False, header=False)